



















































































































































































Gen AI based reserch paper summarization tool with question answering system

In [ ]:
!pip install -q \
    gradio \
    langchain \
    langchain-groq \
    langchain-huggingface \
    langchain-community \
    faiss-cpu \
    pymupdf4llm \
    sentence-transformers
!pip install -q langchain-text-splitters
!!pip install -q langchain-community
!pip install -q nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 21.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source

In [ ]:
import os
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print(' Key loaded from Colab Secrets')
except:
    GROQ_API_KEY = 'paste_your_key_here'  # fallback

LLM_MODEL       = 'llama-3.3-70b-versatile'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('Model:', LLM_MODEL)

 Key loaded from Colab Secrets
Model: llama-3.3-70b-versatile


In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# This replaces your: AutoModelForSeq2SeqLM.from_pretrained("google/long-t5-tglobal-base")
# Instead of loading a model locally, we connect to Groq's API
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=LLM_MODEL
)

# This is EXACTLY your: SentenceTransformer('all-MiniLM-L6-v2')
# Just wrapped in LangChain so FAISS can use it automatically
embedder = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Quick test — make sure both work before going further
test = llm.invoke('Reply with only the word: OK').content
print('LLM test:', test)

test_emb = embedder.embed_query('hello')
print('Embedder test:', len(test_emb), 'dimensions')
# Should print: LLM test: OK
# Should print: Embedder test: 384 dimensions

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM test: OK
Embedder test: 384 dimensions


In [ ]:
import re

def extract_text_from_pdf(pdf_path):
    import pymupdf4llm    # ← move import inside the function
    return pymupdf4llm.to_markdown(pdf_path)


def clean_text(text):
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


def map_heading(heading_text):
    mapping = {
        'abstract':     ['abstract'],
        'introduction': ['introduction', 'background'],
        'literature':   ['literature review', 'related work',
                         'prior work', 'previous work'],
        'methodology':  ['methodology', 'methods', 'approach',
                         'proposed method', 'system design',
                         'proposed approach'],
        'results':      ['results', 'experiments', 'evaluation',
                         'experimental results', 'findings'],
        'discussion':   ['discussion', 'analysis'],
        'conclusion':   ['conclusion', 'conclusions',
                         'future work', 'summary'],
        'references':   ['references', 'bibliography'],
    }
    for standard_name, keywords in mapping.items():
        for keyword in keywords:
            if keyword in heading_text:
                return standard_name
    return heading_text


def detect_sections(markdown_text):
    sections = {}
    current_section = 'unknown'
    current_content = []

    for line in markdown_text.split('\n'):
        if re.match(r'^#{1,3}\s+', line):
            if current_content:
                sections[current_section] = '\n'.join(current_content)
            heading = re.sub(r'^#{1,3}\s+', '', line).strip().lower()
            current_section = map_heading(heading)
            current_content = []
        else:
            current_content.append(line)

    if current_content:
        sections[current_section] = '\n'.join(current_content)

    sections = {k: v for k, v in sections.items() if len(v.strip()) > 80}
    return sections


print('✅ PDF + section functions ready')



✅ PDF + section functions ready


if our research paper does not follow ieee standards

summarization

In [ ]:
import json

def refine_sections(sections, llm):
    raw_names = list(sections.keys())

    # We're asking the LLM to clean up messy heading names
    # The trick is asking for JSON output so we can parse it reliably
    prompt = f"""You are given section headings from a research paper.
Clean and normalize them to standard academic section names.

Rules:
- Map to one of: abstract, introduction, literature, methodology, results, discussion, conclusion, references
- If it does not fit any of those, keep it short and lowercase (max 3 words)
- Remove numbering like 1., 2.1, III. etc.
- Return ONLY a JSON array of strings, same length and order as the input
- No explanation, no extra text, just the JSON array

Input headings:
{json.dumps(raw_names)}

Output:"""

    response = llm.invoke(prompt).content.strip()

    # LLMs sometimes wrap JSON in ```json ... ``` — strip that
    response = re.sub(r'^```json\s*', '', response)
    response = re.sub(r'^```\s*',     '', response)
    response = re.sub(r'\s*```$',     '', response)

    try:
        refined = json.loads(response)
        # Make sure we got back the same number of names
        if isinstance(refined, list) and len(refined) == len(raw_names):
            return refined
    except:
        pass

    # If LLM gave a bad response, just use original names
    print('⚠️ Refinement failed, using original names')
    return raw_names


def split_sections_with_content(full_text, refined_names):
    """
    After refining the names, this function maps each refined name
    back to the actual text content of that section.

    Example:
      refined_names = ['abstract', 'methodology', 'results']
      returns = {
        'abstract':    '...abstract text...',
        'methodology': '...methods text...',
        'results':     '...results text...'
      }
    """
    lines = full_text.split('\n')
    heading_map = {}   # {line_number: refined_name}
    heading_idx = 0

    # First pass: find which lines are headings, map to refined names
    for i, line in enumerate(lines):
        if re.match(r'^#{1,3}\s+', line):
            if heading_idx < len(refined_names):
                heading_map[i] = refined_names[heading_idx]
                heading_idx += 1

    if not heading_map:
        # No headings found — treat whole paper as one section
        return {refined_names[0]: full_text} if refined_names else {'content': full_text}

    # Second pass: collect content between headings
    sections = {}
    current_name  = None
    current_lines = []

    for i, line in enumerate(lines):
        if i in heading_map:
            # Save previous section
            if current_name and current_lines:
                content = '\n'.join(current_lines).strip()
                if len(content) > 80:
                    sections[current_name] = content
            # Start new section
            current_name  = heading_map[i]
            current_lines = []
        else:
            current_lines.append(line)

    # Save the last section
    if current_name and current_lines:
        content = '\n'.join(current_lines).strip()
        if len(content) > 80:
            sections[current_name] = content

    return sections

print('✅ Section refinement ready')

✅ Section refinement ready


In [ ]:
def generate_summary(section_content, llm):
    if not section_content or len(section_content.strip()) < 50:
        return 'No content available for this section.'

    # Truncate very long sections so we don't hit token limits
    content = section_content[:6000]

    # This replaces your entire tokenizer + model.generate() block
    # Instead we just write a clear prompt and send it to Groq
    prompt = f"""You are an expert research paper analyst.
Summarise the following section clearly and in detail.

Guidelines:
- Write 3 to 5 well-formed paragraphs
- Highlight key findings, methods, or arguments
- Use plain English, avoid unnecessary jargon
- Do NOT use bullet points

Section content:
{content}

Summary:"""

    # This one line replaces ~15 lines of your tokenizer/generate code
    return llm.invoke(prompt).content.strip()

print('✅ Summarisation ready')

✅ Summarisation ready


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def create_vector_db(text, embedder):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        separators=['\n\n', '\n', '. ', ' ', ''],
    )
    chunks = splitter.split_text(text)
    chunks = [c for c in chunks if len(c.strip()) > 60]
    print(f'Created {len(chunks)} chunks')

    documents = [Document(page_content=chunk) for chunk in chunks]
    vectordb  = FAISS.from_documents(documents, embedder)
    print(f'Vector store ready — {vectordb.index.ntotal} vectors')
    return vectordb


QA_PROMPT = PromptTemplate(
    input_variables=['context', 'question'],
    template="""You are a research paper assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say: I could not find that in the paper.

Context:
{context}

Question: {question}

Answer:"""
)


def get_qa_chain(vectordb, llm):
    retriever = vectordb.as_retriever(
        search_type='mmr',        # ← change from similarity to mmr
        search_kwargs={
            'k': 8,               # ← retrieve more chunks
            'fetch_k': 20,        # ← consider more candidates
            'lambda_mult': 0.7    # ← balance relevance vs diversity
        }
    )

    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | QA_PROMPT
        | llm
        | StrOutputParser()
    )
    return chain


print('✅ Vector DB + RAG ready')

✅ Vector DB + RAG ready


In [ ]:
# Global variables — using individual globals instead of a dict
# this works reliably across Gradio threads and notebook cells
full_text    = ''
sections     = {}
vector_db    = None
paper_ready  = False


def process_pdf(pdf_path):
    global full_text, sections, vector_db, paper_ready

    try:
        print('Step 1: Extracting text...')
        import pymupdf4llm
        extracted = pymupdf4llm.to_markdown(pdf_path)

        if not extracted or len(extracted.strip()) < 100:
            return [], '❌ Could not extract text from PDF'

        full_text   = extracted
        vector_db   = None
        paper_ready = False
        print(f'  Extracted {len(full_text):,} characters')

        print('Step 2: Detecting sections...')
        raw_sections = detect_sections(full_text)
        print(f'  Found: {list(raw_sections.keys())}')

        if not raw_sections:
            return [], '❌ No sections detected in this PDF'

        print('Step 3: Refining section names with LLM...')
        refined_names = refine_sections(raw_sections, llm)
        print(f'  Refined: {refined_names}')

        print('Step 4: Mapping names to content...')
        sections    = split_sections_with_content(full_text, refined_names)
        paper_ready = True

        print('Step 5: Building vector store...')
        vector_db = create_vector_db(full_text, embedder)

        names = list(sections.keys())
        print(f'\n✅ Done! Sections: {names}')
        return names, f'✅ Paper ready — {len(names)} sections found'

    except Exception as e:
        import traceback; traceback.print_exc()
        return [], f'❌ Error: {str(e)}'


def get_summary(section_name):
    global sections, paper_ready
    if not paper_ready:
        return '⚠️ Upload a paper first.'
    content = sections.get(section_name, '')
    if not content:
        return f'⚠️ No content found for: {section_name}'
    print(f'Summarising: {section_name}...')
    return generate_summary(content, llm)


def answer_question(question, history):
    global full_text, vector_db, paper_ready

    if not paper_ready:
        return '', history + [{"role": "user", "content": question},
                               {"role": "assistant", "content": "⚠️ Upload a paper first."}]

    if vector_db is None:
        vector_db = create_vector_db(full_text, embedder)

    chain    = get_qa_chain(vector_db, llm)
    response = chain.invoke(question)

    history  = history + [
        {"role": "user",      "content": question},
        {"role": "assistant", "content": response}
    ]
    return '', history

print('✅ Pipeline ready')

✅ Pipeline ready


In [ ]:
print(repr(state['full_text'][:500]))

''


In [ ]:
import gradio as gr
import nest_asyncio
nest_asyncio.apply()

def handle_upload(pdf_file):
    global full_text, sections, vector_db, paper_ready

    print(f'handle_upload called with: {type(pdf_file)} — {pdf_file}')  # debug

    if pdf_file is None:
        return gr.update(choices=[], value=None), '⚠️ No file selected.'

    # handle both string path and file object
    pdf_path = pdf_file if isinstance(pdf_file, str) else pdf_file.name
    print(f'Processing path: {pdf_path}')  # debug

    section_list, status = process_pdf(pdf_path)
    first = section_list[0] if section_list else None
    return gr.update(choices=section_list, value=first), status


def handle_summary(section_name):
    return get_summary(section_name) if section_name else '⚠️ Select a section first.'


def handle_chat(message, history):
    _, updated = answer_question(message, history)
    return '', updated


with gr.Blocks(theme=gr.themes.Soft(), title='PaperLens') as demo:

    gr.Markdown('# LARA\n### LLM-Augmented Research Assistant')

    with gr.Row():
        pdf_input  = gr.File(
            label='Upload PDF',
            file_types=['.pdf'],
            type='filepath'   # ← ensures path is passed as string
        )
        with gr.Column():
            upload_btn = gr.Button('🚀 Analyse Paper', variant='primary')
            status_box = gr.Textbox(
                label='Status',
                value='Upload a PDF to begin',
                interactive=False
            )

    gr.Markdown('---')

    with gr.Tabs():

        with gr.TabItem('📑 Section Summaries'):
            with gr.Row():
                with gr.Column(scale=1):
                    section_dd    = gr.Dropdown(label='Select Section', choices=[], interactive=True)
                    summarise_btn = gr.Button('✨ Summarise', variant='primary')
                with gr.Column(scale=2):
                    summary_out   = gr.Textbox(
                        label='Summary',
                        lines=15,
                        interactive=False,
                        show_copy_button=True
                    )

        with gr.TabItem('💬 Ask the Paper'):
            chatbot = gr.Chatbot(height=400, type='messages')
            with gr.Row():
                chat_in  = gr.Textbox(
                    placeholder='Ask anything about the paper...',
                    scale=4,
                    label=''
                )
                send_btn = gr.Button('Send', variant='primary', scale=1)
            gr.Examples(
                examples=[
                    'What is the main contribution?',
                    'What dataset was used?',
                    'What are the key limitations?',
                    'How does the method compare to baselines?',
                ],
                inputs=chat_in
            )

    # wire buttons
    upload_btn.click(
        fn=handle_upload,
        inputs=[pdf_input],
        outputs=[section_dd, status_box]
    )
    summarise_btn.click(
        fn=handle_summary,
        inputs=[section_dd],
        outputs=[summary_out]
    )
    send_btn.click(
        fn=handle_chat,
        inputs=[chat_in, chatbot],
        outputs=[chat_in, chatbot]
    )
    chat_in.submit(
        fn=handle_chat,
        inputs=[chat_in, chatbot],
        outputs=[chat_in, chatbot]
    )

# close any existing demo first to avoid port conflicts
try:
    demo.close()
except:
    pass

demo.launch(share=True, debug=True)  # debug=True shows errors in the UI

/tmp/ipykernel_18024/1557991110.py:31: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='PaperLens') as demo:
/tmp/ipykernel_18024/1557991110.py:67: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400, type='messages')


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cd0a4b114a933095c8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


handle_upload called with: <class 'gradio.utils.NamedString'> — /tmp/gradio/a7218ccb9fef034b9968b92d702fe3bf2f7b04be9f7ca4e8c0a3a45f676477fe/2008.tal-2.5.pdf
Processing path: /tmp/gradio/a7218ccb9fef034b9968b92d702fe3bf2f7b04be9f7ca4e8c0a3a45f676477fe/2008.tal-2.5.pdf
Step 1: Extracting text...
=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=6/7.
OCR on page.number=8/9.

  Extracted 62,261 characters
Step 2: Detecting sections...
  Found: ['**horacio saggion**', 'introduction', '**2. automatic text summarization**', '**3. summa implementation**', '**3.1.** _**gate**_', '**3.2.** _**summa toolkit**_', '**3.3.** _**language resources**_', '**3.4.** _**multi-document summarization in summa**_', '**3.5.** _**ready-made summarization applications in summa**_', 'results', '**5.1.** _**content-based metrics**_', 'conclusion', '**6. learning framework for summarization**', '**7. practical applications and multilinguality**', 'literature', '120 traitement

In [ ]:
!pip install pymupdf4llm transformers torch sentencepiece -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/long-t5-tglobal-base"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Ready on: {device}")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Ready on: cpu


In [ ]:
def summarize_text(text, max_new_tokens=300, min_new_tokens=80):
    prompt = f"summarize: {text}"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=4096,
        truncation=True,
        padding=True,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
def summarize_paper(pdf_path):
    # Step 1: extract
    print("Extracting text...")
    extracted = extract_text_from_pdf(pdf_path)
    full_text = clean_text(extracted["full_text"])

    total_chars = len(full_text)
    target_chars = total_chars // 10        # 10% of paper length
    target_tokens = target_chars // 4       # rough chars → tokens conversion

    # clamp so we don't ask for impossibly short or huge outputs
    max_new_tokens = max(100, min(target_tokens, 1024))
    min_new_tokens = max(50,  max_new_tokens // 2)

    print(f"Paper length : {total_chars:,} chars")
    print(f"Target       : {target_chars:,} chars (~{target_tokens} tokens)")
    print(f"Will generate: {min_new_tokens}–{max_new_tokens} tokens")

    # Step 2: detect and score sections
    print("Detecting sections...")
    sections = detect_sections(full_text)
    scored   = score_sections(sections)
    print(f"Sections found: {[s['section'] for s in scored]}")

    # Step 3: feed more context since we want a longer output
    focused_text = get_important_text(scored, max_chars=12000)
    print(f"Feeding {len(focused_text):,} chars to LongT5...")

    # Step 4: summarize with dynamic length
    summary = summarize_text(
        focused_text,
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
    )

    actual_chars = len(summary)
    print(f"\nSummary      : {actual_chars:,} chars")
    print(f"Coverage     : {actual_chars/total_chars*100:.1f}% of paper")

    return summary, scored

In [ ]:
"""def get_important_text(scored_sections, max_chars=4000):

    important_text = ""
    chars_used     = 0

    for section in scored_sections:
        remaining = max_chars - chars_used

        if remaining <= 0:
            break

        heading         = f"\n\n=== {section['section'].upper()} ===\n"
        content         = section['content'][:remaining]
        important_text += heading + content
        chars_used     += len(content)

    return important_text.strip()"""

In [ ]:
#def score_sections(sections):

   """ importance_scores = {
        "abstract":      10,
        "conclusion":    9,
        "results":       8,
        "introduction":  7,
        "methodology":   7,
        "discussion":    6,
        "literature":    3,
        "unknown":       2,
        "references":    0
    }

    scored = []

    for section_name, content in sections.items():
        score = importance_scores.get(section_name, 2)

        if score == 0:
            continue
        if len(content.strip()) < 100:
            continue

        scored.append({
            "section": section_name,
            "content": content,
            "score":   score,
            "length":  len(content)
        })

    # Sort highest score first
    scored.sort(key=lambda x: x["score"], reverse=True)

    return scored """




IndentationError: unexpected indent (4028161408.py, line 3)

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"File: {pdf_path}")

summary, scored = summarize_paper(pdf_path)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(summary)

Saving 2008.tal-2.5.pdf to 2008.tal-2.5.pdf
File: 2008.tal-2.5.pdf
Extracting text...
=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=6/7.
OCR on page.number=8/9.

=== Document parser messages ===
                                                                                    Using Tesseract for OCR processing.
OCR on page.number=6/7.
OCR on page.number=8/9.

Paper length : 62,257 chars
Target       : 6,225 chars (~1556 tokens)
Will generate: 512–1024 tokens
Detecting sections...
Sections found: ['conclusion', 'results', 'introduction', 'literature', '**horacio saggion**', '**2. automatic text summarization**', '**3. summa implementation**', '**3.1.** _**gate**_', '**3.2.** _**summa toolkit**_', '**3.3.** _**language resources**_', '**3.4.** _**multi-document summarization in summa**_', '**3.5.** _**ready-made summarization applications in summa**_', '**5.1.** _**content-based metrics**_', '**6. learning framework for summarization**', '**7. 

In [ ]:
!pip install sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.4 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# load a small but powerful embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # only 80MB

def build_vector_store(full_text, chunk_size=150, overlap=50):
    words = full_text.split()
    chunks = []

    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:
            chunks.append(chunk)
        i += (chunk_size - overlap)  # slide forward with overlap

    print(f"Created {len(chunks)} chunks from {len(words)} words")

    print("Embedding chunks...")
    embeddings = embedder.encode(chunks, show_progress_bar=True)
    embeddings = np.array(embeddings, dtype='float32')

    # normalize vectors — improves similarity matching significantly
    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(embeddings.shape[1])  # IP = inner product, better than L2 after normalizing
    index.add(embeddings)

    print(f"Vector store ready — {index.ntotal} vectors stored")
    return index, chunks

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def retrieve(question, index, chunks, top_k=8):
    # embed the question using the same model
    question_vector = embedder.encode([question])
    question_vector = np.array(question_vector, dtype='float32')

    # find top_k most similar chunks
    distances, indices = index.search(question_vector, top_k)

    retrieved = []
    for i, idx in enumerate(indices[0]):
        retrieved.append({
            "chunk": chunks[idx],
            "score": float(distances[0][i])  # lower = more similar
        })

    return retrieved

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# load flan-t5 manually instead of using pipeline
print("Loading QA model...")
qa_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
qa_model_t5  = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

qa_device = "cuda" if torch.cuda.is_available() else "cpu"
qa_model_t5 = qa_model_t5.to(qa_device)
print(f"QA model ready on: {qa_device}")

def answer_question(question, index, chunks):
    # step 1: retrieve relevant chunks
    retrieved = retrieve(question, index, chunks, top_k=3)

    # step 2: build context
    context = "\n\n".join([r["chunk"] for r in retrieved])

    # step 3: build prompt
    prompt = f"""Answer the question based only on the context below.
If the answer is not in the context, say "I couldn't find that in the paper."

Context:
{context}

Question: {question}

Answer:"""

    # step 4: tokenize and generate
    inputs = qa_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(qa_device)

    with torch.no_grad():
        output_ids = qa_model_t5.generate(
            **inputs,
            max_new_tokens=200,
            num_beams=4,
            early_stopping=True
        )

    return qa_tokenizer.decode(output_ids[0], skip_special_tokens=True)

Loading QA model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


QA model ready on: cpu


In [ ]:
# after uploading and extracting your PDF, run this once:
extracted = extract_text_from_pdf(pdf_path)
full_text = clean_text(extracted["full_text"])

index, chunks = build_vector_store(full_text)
print("Ready to answer questions!")

# then ask anything:
questions = [
    "What is the main contribution of this paper?",
    "What dataset did the authors use?",
    "What were the main results?",
    "What are the limitations mentioned?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {answer_question(q, index, chunks)}")

=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                Using Tesseract for OCR processing.
OCR on page.number=6/7.
OCR on page.number=8/9.

=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                                                    Using Tesseract for OCR processing.
OCR on page.number=6/7.
OCR on page.number=8/9.

Created 

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vector store ready — 95 vectors stored
Ready to answer questions!

Q: What is the main contribution of this paper?
A: document has also informed content selection

Q: What dataset did the authors use?
A: I couldn't find that in the paper

Q: What were the main results?
A: I couldn't find that in the paper

Q: What are the limitations mentioned?
A: I couldn't find that in the paper


In [ ]:
test_question = "What is the main contribution of this paper?"
retrieved = retrieve(test_question, index, chunks, top_k=5)

print("QUESTION:", test_question)
print(f"Total chunks: {len(chunks)}")
print("\nTOP 5 RETRIEVED CHUNKS:\n")
for i, r in enumerate(retrieved):
    print(f"--- Chunk {i+1} | score: {r['score']:.3f} ---")
    print(r['chunk'])
    print()

QUESTION: What is the main contribution of this paper?
Total chunks: 95

TOP 5 RETRIEVED CHUNKS:

--- Chunk 1 | score: 0.395 ---
document has also informed content selection strategies (Edmundson, 1969; Lin and Hovy, 1997). News wire and similar texts have a pyramidal discourse structure where relevant/new information (the key events) is usually expressed in the leading paragraph, and therefore sentences from the leading paragraph are considered relevant. In scientific papers, sections such as the introduction and the conclusion report on the objectives and findings of the research, so sentences contained in those sections can be considered as important. Because titles of articles, especially scientific ones, sometimes state the theme or subject dealt with in the article, one may consider that sentences containing terms from the title are relevant for a summary (Edmundson, 1969). The presence of specific formulaic expressions, especially in the scientific and legal domains, has also be

In [ ]:
# Cell 1 — rebuild with specter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Loading allenai-specter...")
embedder = SentenceTransformer('allenai-specter')
print("Done!")

Loading allenai-specter...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Done!


In [ ]:
# Cell 2 — rebuild vector store (same function, just re-run it)
index, chunks = build_vector_store(full_text)

Created 95 chunks from 9409 words
Embedding chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vector store ready — 95 vectors stored


In [ ]:
# Cell 3 — improve the prompt to be more direct
def answer_question(question, index, chunks, top_k=5):
    retrieved = retrieve(question, index, chunks, top_k=top_k)

    # print scores so you can monitor improvement
    scores = [round(r['score'], 3) for r in retrieved]
    print(f"Retrieval scores: {scores}")

    context = "\n\n".join([r["chunk"] for r in retrieved])

    # more explicit prompt works better with FLAN-T5
    prompt = f"""You are a research paper assistant.
Read the context below and answer the question directly and specifically.
If the exact answer is not stated, summarize what the context says about the topic.
Do not say "I couldn't find that" unless the context is completely unrelated.

Context:
{context}

Question: {question}

Specific answer:"""

    inputs = qa_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(qa_device)

    with torch.no_grad():
        output_ids = qa_model_t5.generate(
            **inputs,
            max_new_tokens=250,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return qa_tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
# Cell 4 — also ask questions that match THIS paper's actual content
questions = [
    "What is this paper about?",
    "What is the SUMMA toolkit?",
    "What features are used for sentence selection?",
    "What is extractive summarization?",
    "What is tf-idf and how is it used?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {answer_question(q, index, chunks)}")


Q: What is this paper about?
Retrieval scores: [16.849, 16.823, 16.611, 16.593, 16.565]
A: I couldn't find that

Q: What is the SUMMA toolkit?
Retrieval scores: [17.436, 17.395, 17.139, 17.065, 16.878]
A: the document.

Q: What features are used for sentence selection?
Retrieval scores: [19.298, 19.142, 19.118, 18.989, 18.958]
A: I couldn't find that

Q: What is extractive summarization?
Retrieval scores: [19.731, 19.537, 19.401, 19.388, 19.381]
A: summarization tools based on GATE

Q: What is tf-idf and how is it used?
Retrieval scores: [17.394, 17.228, 17.131, 17.113, 17.089]
A: Centroid computation and sentence-centroid similarity computation


In [ ]:
import pymupdf4llm

def summarize_paper(pdf_path):
    """
    Full pipeline: PDF → sections → scored text → summary.
    """
    # Step 1: extract markdown from PDF
    print("Extracting text...")
    md_text = pymupdf4llm.to_markdown(pdf_path)

    # Step 2: your section detection code
    print("Detecting sections...")
    sections = detect_sections(md_text)
    scored   = score_sections(sections)

    print(f"Found sections: {[s['section'] for s in scored]}")

    # Step 3: build focused input using your budget function
    focused_text = get_important_text(scored, max_chars=8000)

    print(f"Feeding {len(focused_text)} chars to LongT5...")

    # Step 4: summarize
    summary = summarize_text(focused_text)

    return summary, scored   # return scored too so you can inspect sections

In [ ]:
from google.colab import files

# Upload your PDF
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Run the pipeline
summary, sections = summarize_paper(pdf_path)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(summary)

In [ ]:
#for testing our code
from google.colab import files

# This opens a file picker in Colab
uploaded = files.upload()

# Get the filename that was uploaded
filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

# Now test directly with that file
result   = extract_text_from_pdf(filename)
cleaned  = clean_text(result["full_text"])
sections = detect_sections(cleaned)
scored   = score_sections(sections)

print("\nSections ranked by importance:")
print("-" * 40)
for s in scored:
    print(f"Score {s['score']:2d}  →  {s['section']:<15} ({s['length']} chars)")

In [ ]:
!pip uninstall -y transformers tokenizers

In [ ]:
!rm -rf ~/.cache/huggingface

In [ ]:
!pip install transformers==4.41.2

longt5

In [ ]:
!pip install transformers torch sentencepiece

In [ ]:
from transformers import pipeline
import torch

print("Loading LongT5 model...")
print("First time takes 3-4 mins — grab a coffee! ☕")

device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("✅ GPU detected — model will run fast!")
else:
    print("⚠️ No GPU — running on CPU, will be slower")

summarizer = pipeline(
    "summarization",
    model  = "pszemraj/long-t5-tglobal-base-16384-book-summary",
    device = device
)

print("✅ LongT5 model loaded and ready!")

In [ ]:
def generate_summary(full_text):
    """
    Full hybrid summarization pipeline:

    YOUR CODE:
      Step 1 — Detect sections from markdown headings
      Step 2 — Score sections by importance
      Step 3 — Build focused text from top sections

    AI MODEL:
      Step 4 — LongT5 generates fluent summary
      (handles much longer input than BART)
    """

    print("\n--- Starting Summarization Pipeline ---")

    # Step 1 — Detect sections
    sections = detect_sections(full_text)
    print(f"Step 1 ✅ Detected {len(sections)} sections:")
    for name in sections:
        print(f"         → {name}")

    # Step 2 — Score sections
    scored = score_sections(sections)
    print(f"\nStep 2 ✅ Sections ranked:")
    for s in scored:
        print(f"         Score {s['score']} → {s['section']}")

    # Step 3 — Get important text
    # LongT5 handles up to ~48000 chars so we increase budget
    important_text = get_important_text(scored, max_chars=8000)
    print(f"\nStep 3 ✅ Important text: {len(important_text)} chars")

    if len(important_text) < 100:
        return "Could not generate summary — not enough text extracted."

    # Step 4 — LongT5 summarizes
    print("\nStep 4 ⏳ LongT5 generating summary...")

    # No trimming needed — LongT5 handles long inputs natively
    result = summarizer(
        important_text,
        max_length = 350,   # longer summary than BART
        min_length = 100,
        do_sample  = False
    )

    summary = result[0]['summary_text']
    print("Step 4 ✅ Summary generated!")
    print("\n--- Pipeline Complete ---\n")

    return summary

In [ ]:
# BART — had to trim aggressively
important_text = get_important_text(scored, max_chars=4000)
bart_input     = important_text[:3000]  # extra trim

# LongT5 — can handle much more
important_text = get_important_text(scored, max_chars=8000)
# no trimming needed — pass full text directly

bart cont

In [ ]:
from transformers import pipeline
from transformers.pipelines import PIPELINE_REGISTRY

print(PIPELINE_REGISTRY.get_supported_tasks())

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device
)

print("✅ Working!")

In [ ]:
html_content = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title></title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

    <div class="container">
        <h1>Meet LARA your literature analysis and research assistant </h1>
        <p class="subtitle">Upload a research paper and get an instant summary</p>

        <!-- Upload Section -->
        <div class="upload-box" id="uploadBox">
            <div class="upload-icon">📄</div>
            <p>Drag and drop your PDF here</p>
            <p class="or">or</p>

            <!-- This is the actual file input -->
            <!-- "accept" tells browser only allow PDFs -->
            <input type="file" id="fileInput" accept=".pdf" hidden>

            <!-- We hide the ugly default input and use our own button -->
            <button class="btn" onclick="document.getElementById('fileInput').click()">
                Choose PDF
            </button>

            <!-- Shows the selected filename -->
            <p id="fileName" class="file-name"></p>
        </div>

        <!-- Upload Button -->
        <button class="btn btn-primary" id="uploadBtn" disabled>
            Upload and Extract Text
        </button>

        <!-- Loading spinner - hidden by default -->
        <div class="loader" id="loader" style="display:none">
            <div class="spinner"></div>
            <p>Reading your PDF...</p>
        </div>

        <!-- Results Section - hidden by default -->
        <div class="results" id="results" style="display:none">
            <h2>Extracted Text Preview</h2>
            <div class="info-bar">
                <span id="pageCount"></span>
                <span id="charCount"></span>
            </div>
            <div class="text-preview" id="textPreview"></div>
        </div>
    </div>

    <script src="/static/script.js"></script>
</body>
</html>'''

# Write this to our static folder
with open("research_analyzer/static/index.html", "w") as f:
    f.write(html_content)

print("index.html created")

In [ ]:
css_content = '''
/* Basic reset - removes default browser spacing */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

  body {
    font-family: 'Segoe UI', sans-serif;
    background: radial-gradient(circle at top, #1a1a2e, #0f0f1a);
    color: #e5e7eb;
    min-height: 100vh;
    display: flex;
    align-items: center;
    justify-content: center;
}



.container {
    background: rgba(255, 255, 255, 0.05);
    backdrop-filter: blur(14px);
    padding: 2.5rem;
    border-radius: 16px;
    width: 90%;
    max-width: 700px;
    box-shadow: 0 0 40px rgba(79, 70, 229, 0.25);
    border: 1px solid rgba(255,255,255,0.08);
}


h1 {
    font-size: 2rem;
    color: #c7d2fe;
}

.subtitle {
    color: #a5b4fc;
}



/* Upload box styling */
.upload-box {
    border: 2px dashed rgba(165, 180, 252, 0.4);
    background: rgba(255,255,255,0.02);
}

.upload-box.drag-over {
    border-color: #818cf8;
    background: rgba(99, 102, 241, 0.08);
}



.upload-icon {
    font-size: 3rem;
    margin-bottom: 1rem;
}

.or {
    color: #999;
    margin: 0.75rem 0;
    font-size: 0.9rem;
}

.file-name {
    margin-top: 1rem;
    color: #4f46e5;
    font-weight: 500;
    font-size: 0.9rem;
}

/* Button styles */
.btn {
    padding: 0.6rem 1.4rem;
    border: 2px solid #4f46e5;
    border-radius: 8px;
    background: white;
    color: #4f46e5;
    font-size: 0.95rem;
    cursor: pointer;
    transition: all 0.2s;
}

.btn:hover {
    background: #4f46e5;
    color: white;
}

.btn-primary {
    width: 100%;
    padding: 0.9rem;
    background: linear-gradient(135deg, #6366f1, #8b5cf6);
    color: white;
    border: none;
    border-radius: 10px;
    font-size: 1rem;
    cursor: pointer;
}

.btn-primary:hover {
    box-shadow: 0 0 20px rgba(139, 92, 246, 0.6);
    transform: translateY(-1px);
}


/* Disabled state - when no file selected */
.btn-primary:disabled {
    background: #c0c8d8;
    cursor: not-allowed;
    transform: none;
}

/* Loading spinner */
.loader {
    text-align: center;
    padding: 2rem;
    color: #666;
}

.spinner {
    width: 40px;
    height: 40px;
    border: 3px solid #e0e0e0;
    border-top-color: #4f46e5;
    border-radius: 50%;
    animation: spin 0.8s linear infinite;
    margin: 0 auto 1rem;
}

@keyframes spin {
    to { transform: rotate(360deg); }
}

/* Results section */
.results {
    margin-top: 1.5rem;
    animation: fadeIn 0.3s ease;
}

@keyframes fadeIn {
    from { opacity: 0; transform: translateY(10px); }
    to   { opacity: 1; transform: translateY(0); }
}

.results h2 {
    font-size: 1.2rem;
    margin-bottom: 0.75rem;
    color: #1a1a2e;
}

.info-bar {
    display: flex;
    gap: 1.5rem;
    margin-bottom: 1rem;
    font-size: 0.85rem;
    color: #666;
}

.text-preview {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 10px;
    padding: 1.25rem;
    font-size: 0.9rem;
    line-height: 1.7;
    max-height: 300px;
    overflow-y: auto;
    white-space: pre-wrap;  /* preserves line breaks */
}
'''

with open("research_analyzer/static/style.css", "w") as f:
    f.write(css_content)

print(" style.css created")

In [ ]:
js_content = '''
// Grab references to all the HTML elements we need
const fileInput   = document.getElementById("fileInput");
const uploadBox   = document.getElementById("uploadBox");
const fileName    = document.getElementById("fileName");
const uploadBtn   = document.getElementById("uploadBtn");
const loader      = document.getElementById("loader");
const results     = document.getElementById("results");
const textPreview = document.getElementById("textPreview");
const pageCount   = document.getElementById("pageCount");
const charCount   = document.getElementById("charCount");

// ─── EVENT 1: User selects a file ───────────────────────────
// This fires when user picks a file through the file picker
fileInput.addEventListener("change", function() {
    const file = fileInput.files[0];  // get the first selected file

    if (file) {
        // Show the filename so user knows what they selected
        fileName.textContent = "Selected: " + file.name;

        // Enable the upload button now that we have a file
        uploadBtn.disabled = false;
    }
});

// ─── EVENT 2: Drag and Drop ──────────────────────────────────
// When user drags a file OVER the upload box
uploadBox.addEventListener("dragover", function(e) {
    e.preventDefault();  // stops browser from opening the file
    uploadBox.classList.add("drag-over");  // adds blue border via CSS
});

// When user drags the file OUT without dropping
uploadBox.addEventListener("dragleave", function() {
    uploadBox.classList.remove("drag-over");
});

// When user actually DROPS the file
uploadBox.addEventListener("drop", function(e) {
    e.preventDefault();
    uploadBox.classList.remove("drag-over");

    // Get the dropped file
    const file = e.dataTransfer.files[0];

    // Make sure it is a PDF
    if (file && file.type === "application/pdf") {
        // Manually assign it to the file input
        // This is a trick - we copy the dropped file into the input
        const dataTransfer = new DataTransfer();
        dataTransfer.items.add(file);
        fileInput.files = dataTransfer.files;

        // Show filename and enable button
        fileName.textContent = "Selected: " + file.name;
        uploadBtn.disabled = false;
    } else {
        alert("Please drop a PDF file");
    }
});

// ─── EVENT 3: User clicks Upload button ─────────────────────
uploadBtn.addEventListener("click", async function() {

    const file = fileInput.files[0];
    if (!file) return;

    // Show loading, hide results
    loader.style.display = "block";
    results.style.display = "none";
    uploadBtn.disabled = true;

    // FormData is how we package a file to send to Flask
    // Think of it like putting the file in an envelope
    const formData = new FormData();
    formData.append("pdf", file);  // "pdf" is the key Flask will look for

    try {
        // Send the file to Flask
        // fetch() is JavaScript's way of making HTTP requests
        const response = await fetch("/upload", {
            method: "POST",     // POST means we are SENDING data
            body: formData      // the envelope containing our file
        });

        // Parse the JSON response Flask sends back
        const data = await response.json();

        if (data.success) {
            // Show the results
            pageCount.textContent   = "📄 Pages: "      + data.total_pages;
            charCount.textContent   = "📝 Characters: " + data.total_chars;
            textPreview.textContent = data.preview;

            results.style.display = "block";
        } else {
            alert("Error: " + data.error);
        }

    } catch (error) {
        alert("Something went wrong: " + error.message);

    } finally {
        // Always hide loader and re-enable button when done
        loader.style.display  = "none";
        uploadBtn.disabled    = false;
    }
});
'''

with open("research_analyzer/static/script.js", "w") as f:
    f.write(js_content)

print("script.js created")

In [ ]:
from flask import Flask, request, jsonify, send_from_directory
from pyngrok import ngrok
import threading
import os

app = Flask(__name__, static_folder="research_analyzer/static")

@app.route("/")
def home():
    return send_from_directory("research_analyzer/static", "index.html")

@app.route("/upload", methods=["POST"])
def upload():
    if "pdf" not in request.files:
        return jsonify({"success": False, "error": "No file received"})

    file = request.files["pdf"]

    if not file.filename.endswith(".pdf"):
        return jsonify({"success": False, "error": "Please upload a PDF"})

    temp_path = f"research_analyzer/temp_{file.filename}"
    file.save(temp_path)

    try:
        result = extract_text_from_pdf(temp_path)
        clean  = clean_text(result["full_text"])

        return jsonify({
            "success": True,
            "total_pages": result["total_pages"],
            "total_chars": len(clean),
            "preview": clean[:3000]
        })

    except Exception as e:
        return jsonify({"success": False, "error": str(e)})

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


# ✅ CHANGE 1: Use different port
PORT = 5001

def run_flask():
    app.run(port=PORT)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()


# ✅ CHANGE 2: Force pyngrok to reinstall properly
ngrok.kill()  # clears old broken session

# ⚠️ Replace with your actual token (keep it private!)
ngrok.set_auth_token("32Hwcynr8DwYZFoeHJUtdldgbBT_7stvMQXgr3bUf2vLAy46Y")

public_url = ngrok.connect(PORT)

print(f"✅ App is live at: {public_url}")

In [ ]:
from flask import Flask, request, jsonify, send_from_directory
from pyngrok import ngrok
import threading
import os

app = Flask(__name__, static_folder="research_analyzer/static")

@app.route("/")
def home():
    return send_from_directory("research_analyzer/static", "index.html")

@app.route("/upload", methods=["POST"])
def upload():
    if "pdf" not in request.files:
        return jsonify({"success": False, "error": "No file received"})

    file = request.files["pdf"]

    if not file.filename.endswith(".pdf"):
        return jsonify({"success": False, "error": "Please upload a PDF"})

    temp_path = f"research_analyzer/temp_{file.filename}"
    file.save(temp_path)

    try:
        result = extract_text_from_pdf(temp_path)
        clean  = clean_text(result["full_text"])

        return jsonify({
            "success":     True,
            "total_pages": result["total_pages"],
            "total_chars": len(clean),
            "preview":     clean[:3000]
        })

    except Exception as e:
        return jsonify({"success": False, "error": str(e)})

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

# Start Flask and ngrok
def run_flask():
    app.run(port=5000)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()
ngrok.set_auth_token("32Hwcynr8DwYZFoeHJUtdldgbBT_7stvMQXgr3bUf2vLAy46Y")
public_url = ngrok.connect(5000)
print(f"✅ App is live at: {public_url}")
print("Open this URL in your browser!")

In [ ]:
!pip install pyngrok --upgrade
!apt-get install unzip
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -o ngrok-v3-stable-linux-amd64.zip

In [ ]:
!./ngrok config add-authtoken YOUR_TOKEN
!./ngrok http 5001

In [ ]:
def generate_summary(full_text):


    print("\n--- Starting Summarization Pipeline ---")

    sections = detect_sections(full_text)
    print(f" {len(sections)} sections:")
    for name in sections:
        print(f"         → {name}")


    scored = score_sections(sections)
    print(f"\n Sections ranked:")
    for s in scored:
        print(f"         Score {s['score']} → {s['section']}")


    important_text = get_important_text(scored, max_chars=4000)
    print(f"\nImportant text: {len(important_text)} chars")


    if len(important_text) < 100:
        return "Could not generate summary — not enough text extracted."


    print("\nStep 4 ⏳ BART generating summary...")

    # BART has a token limit — if text is still too long
    # we trim it to be safe (1024 tokens ≈ 3000 characters)
    bart_input = important_text[:3000]

    result = summarizer(
      bart_input,
      max_length=200,
      min_length=60,
      num_beams=4,
      length_penalty=2.0,
      early_stopping=True,
      do_sample=False
)


    summary = result[0]['summary_text']
    print("Summary generated!")
    print("\n--- Pipeline Complete ---\n")

    return summary


In [ ]:

result   = extract_text_from_pdf(filename)

cleaned  = clean_text(result["full_text"])


summary = generate_summary(cleaned)

print("\n" + "="*60)
print("GENERATED SUMMARY")
print("="*60 + "\n")
print(summary)

hosting

In [ ]:
html_content = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Research Paper Analyzer</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>
    <div class="container">

        <h1>Research Paper Analyzer</h1>
        <p class="subtitle">Upload a research paper and get an instant summary</p>

        <!-- Upload Section -->
        <div class="upload-box" id="uploadBox">
            <div class="upload-icon">📄</div>
            <p>Drag and drop your PDF here</p>
            <p class="or">or</p>
            <input type="file" id="fileInput" accept=".pdf" hidden>
            <button class="btn"
                onclick="document.getElementById('fileInput').click()">
                Choose PDF
            </button>
            <p id="fileName" class="file-name"></p>
        </div>

        <!-- Upload Button -->
        <button class="btn btn-primary" id="uploadBtn" disabled>
            Upload and Extract Text
        </button>

        <!-- Loading indicator -->
        <div class="loader" id="loader" style="display:none">
            <div class="spinner"></div>
            <p id="loaderText">Reading your PDF...</p>
        </div>

        <!-- After upload success -->
        <div class="results" id="extractResults" style="display:none">
            <h2>✅ Paper Uploaded Successfully</h2>
            <div class="info-bar">
                <span id="pageCount"></span>
                <span id="charCount"></span>
            </div>
            <button class="btn btn-primary" id="summarizeBtn">
                ✨ Generate Summary
            </button>
        </div>

        <!-- Summary output -->
        <div class="results" id="summaryResults" style="display:none">
            <h2>📝 Summary</h2>
            <div class="summary-box" id="summaryText"></div>
        </div>

    </div>
    <script src="/static/script.js"></script>
</body>
</html>'''

with open("research_analyzer/static/index.html", "w") as f:
    f.write(html_content)

print("✅ index.html updated!")

In [ ]:
summary_css = '''
.summary-box {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-left: 4px solid #4f46e5;
    border-radius: 10px;
    padding: 1.5rem;
    font-size: 0.95rem;
    line-height: 1.8;
    color: #1a1a2e;
    margin-top: 1rem;
    white-space: pre-wrap;
}
'''

with open("research_analyzer/static/style.css", "a") as f:
    f.write(summary_css)

print("✅ CSS updated!")

In [ ]:
js_content = '''
const fileInput      = document.getElementById("fileInput");
const uploadBox      = document.getElementById("uploadBox");
const fileName       = document.getElementById("fileName");
const uploadBtn      = document.getElementById("uploadBtn");
const loader         = document.getElementById("loader");
const loaderText     = document.getElementById("loaderText");
const extractResults = document.getElementById("extractResults");
const summarizeBtn   = document.getElementById("summarizeBtn");
const summaryResults = document.getElementById("summaryResults");
const summaryText    = document.getElementById("summaryText");
const pageCount      = document.getElementById("pageCount");
const charCount      = document.getElementById("charCount");

// ── File selection ───────────────────────────────────────────
fileInput.addEventListener("change", function() {
    const file = fileInput.files[0];
    if (file) {
        fileName.textContent = "Selected: " + file.name;
        uploadBtn.disabled   = false;
    }
});

// ── Drag and drop ────────────────────────────────────────────
uploadBox.addEventListener("dragover", function(e) {
    e.preventDefault();
    uploadBox.classList.add("drag-over");
});

uploadBox.addEventListener("dragleave", function() {
    uploadBox.classList.remove("drag-over");
});

uploadBox.addEventListener("drop", function(e) {
    e.preventDefault();
    uploadBox.classList.remove("drag-over");

    const file = e.dataTransfer.files[0];
    if (file && file.type === "application/pdf") {
        const dt             = new DataTransfer();
        dt.items.add(file);
        fileInput.files      = dt.files;
        fileName.textContent = "Selected: " + file.name;
        uploadBtn.disabled   = false;
    } else {
        alert("Please drop a PDF file");
    }
});

// ── Upload ───────────────────────────────────────────────────
// ⚠️ NO custom headers here — letting browser set them automatically
// When sending FormData the browser must set Content-Type itself
// Adding headers manually breaks the file upload
uploadBtn.addEventListener("click", async function() {
    const file = fileInput.files[0];
    if (!file) return;

    showLoader("Reading your PDF...");
    extractResults.style.display = "none";
    summaryResults.style.display = "none";
    uploadBtn.disabled           = true;

    const formData = new FormData();
    formData.append("pdf", file);

    try {
        const response = await fetch("/upload", {
            method: "POST",
            body:   formData
            // ← NO headers here — this was the bug!
        });

        // Read as text first to debug if JSON parsing fails
        const text = await response.text();

        let data;
        try {
            data = JSON.parse(text);
        } catch(e) {
            alert("Upload error — server sent:\n\n" + text.slice(0, 300));
            return;
        }

        if (data.success) {
            pageCount.textContent        = "📄 Pages: "      + data.total_pages;
            charCount.textContent        = "📝 Characters: " + data.total_chars;
            extractResults.style.display = "block";
        } else {
            alert("Upload failed: " + data.error);
        }

    } catch (err) {
        alert("Something went wrong: " + err.message);
    } finally {
        hideLoader();
        uploadBtn.disabled = false;
    }
});

// ── Summarize ────────────────────────────────────────────────
summarizeBtn.addEventListener("click", async function() {
    showLoader("Analyzing paper... this may take 30 seconds ⏳");
    summaryResults.style.display = "none";
    summarizeBtn.disabled        = true;

    try {
        const response = await fetch("/summarize", {
            method:  "POST",
            headers: { "bypass-tunnel-reminder": "true" }
        });

        const text = await response.text();

        let data;
        try {
            data = JSON.parse(text);
        } catch(e) {
            alert("Summarize error — server sent:\n\n" + text.slice(0, 300));
            return;
        }

        if (data.success) {
            summaryText.textContent      = data.summary;
            summaryResults.style.display = "block";
            summaryResults.scrollIntoView({ behavior: "smooth" });
        } else {
            alert("Summarize failed: " + data.error);
        }

    } catch (err) {
        alert("Something went wrong: " + err.message);
    } finally {
        hideLoader();
        summarizeBtn.disabled = false;
    }
});

// ── Helpers ──────────────────────────────────────────────────
function showLoader(message) {
    loaderText.textContent = message;
    loader.style.display   = "block";
}

function hideLoader() {
    loader.style.display = "none";
}
'''

with open("research_analyzer/static/script.js", "w") as f:
    f.write(js_content)

print("✅ script.js fixed!")

In [ ]:
from flask import Flask, request, jsonify, send_from_directory
import threading
import os
import time
import subprocess
import requests as req

app = Flask(__name__, static_folder="research_analyzer/static")

# ── Route 1: Homepage ────────────────────────────────────────
@app.route("/")
def home():
    return send_from_directory("research_analyzer/static", "index.html")

# ── Route 2: Test ────────────────────────────────────────────
@app.route("/test", methods=["GET"])
def test():
    return jsonify({
        "flask":       "working",
        "has_text":    "CURRENT_TEXT" in app.config,
        "text_length": len(app.config.get("CURRENT_TEXT", ""))
    })

# ── Route 3: Upload PDF ──────────────────────────────────────
@app.route("/upload", methods=["POST"])
def upload():
    if "pdf" not in request.files:
        return jsonify({"success": False, "error": "No file received"})

    file = request.files["pdf"]

    if not file.filename.endswith(".pdf"):
        return jsonify({"success": False, "error": "Please upload a PDF"})

    temp_path = f"research_analyzer/temp_{file.filename}"
    file.save(temp_path)

    app.config["CURRENT_PDF"] = temp_path

    try:
        result = extract_text_from_pdf(temp_path)
        clean  = clean_text(result["full_text"])

        app.config["CURRENT_TEXT"] = clean

        # Confirm it was stored
        print(f"✅ Text stored in app.config: {len(clean)} chars")

        return jsonify({
            "success":     True,
            "total_pages": result["total_pages"],
            "total_chars": len(clean),
            "preview":     clean[:3000]
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"success": False, "error": str(e)})

# ── Route 4: Summarize ───────────────────────────────────────
@app.route("/summarize", methods=["POST"])
def summarize():
    text = app.config.get("CURRENT_TEXT")

    print(f"Summarize called — text length: {len(text) if text else 'NONE'}")

    if not text:
        return jsonify({
            "success": False,
            "error":   "No paper uploaded yet. Please upload a PDF first."
        })

    try:
        summary = generate_summary(text)
        return jsonify({
            "success": True,
            "summary": summary
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"success": False, "error": str(e)}), 200

# ── Start Flask ──────────────────────────────────────────────
def run_flask():
    app.run(port=5000)

thread        = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

time.sleep(2)
print("✅ Flask running!")

# ── Get tunnel password ──────────────────────────────────────
tunnel_password = req.get("https://loca.lt/mytunnelpassword").text.strip()
print(f"🔑 Tunnel password: {tunnel_password}")

# ── Start localtunnel ────────────────────────────────────────
os.system("npm install -g localtunnel")
tunnel   = subprocess.Popen(
    ["lt", "--port", "5000"],
    stdout = subprocess.PIPE,
    stderr = subprocess.PIPE
)
url_line = tunnel.stdout.readline().decode("utf-8").strip()
print(f"✅ App live at: {url_line}")
print(f"\n📋 Steps:")
print(f"   1. Open URL above")
print(f"   2. Enter password: {tunnel_password}")
print(f"   3. Upload a PDF")
print(f"   4. Click Generate Summary")


In [ ]:
import time

print("Watching for 20 seconds... upload your PDF now!")
print("-" * 40)

for i in range(20):
    time.sleep(1)
    text = app.config.get("CURRENT_TEXT")

    if text:
        print(f"✅ Upload detected after {i+1} seconds!")
        print(f"   Text length: {len(text)} chars")
        break
    else:
        print(f"   {i+1}s — nothing yet...")

In [ ]:
with open("research_analyzer/static/script.js", "r") as f:
    lines = f.readlines()

# Print lines around line 80
for i, line in enumerate(lines[75:85], start=76):
    print(f"{i}: {line}", end="")